# B1 — `C.Oc138k.compact.e3 → Oc`: Model 2 on a chemically pretrained base

Model 2 has so far used `t5-small`, a general-domain checkpoint that has never seen a molecule,
while three of its four output fields are chemistry: solvent and catalyst are SMILES strings. The
obvious base to try instead is the one Model 1's clean line uses, `sagawa/CompoundT5` — span-MLM
over 24M ZINC20 molecules. It has never seen a *reaction*, so the ORD-derived conditions test
stays leak-free by the same argument that makes the USPTO-50K line clean.

**Why this needs a format change first.** An earlier attempt to put a chemical base under Model 2
(ReactionT5, `git show c38a15e`) produced 0.0% parseable output and gibberish like
`"sole nC(Cl )Cl clsno spe ce e pe rre ce"`. The cause was the target, not the base: field names
and braces are English, and these vocabularies hold SMILES. Measured over 2000 real training rows:

| base | target | mean tokens | `<unk>` per row |
|---|---|---|---|
| t5-small | JSON | 60.9 | 2.0 |
| t5-small | compact | 20.7 | **0** |
| CompoundT5 | JSON | 90.9 | **50.1** |
| CompoundT5 | compact | 23.1 | 8.7 → 0 after vocabulary repair |

So this run changes two things at once — base and target serialization — and both changes are
representational. `--target-format compact` writes `CO|[Pd]|25.0|85.0` instead of the JSON object;
`|` occurs in none of the 152,270 condition rows, and a round-trip test over the whole test set
recovers every field exactly. Numeric fields stay numbers, so every metric in RESULTS.md remains
directly comparable to the t5-small rows.

## What the first attempt got wrong

Version 2 of this notebook finished but scored below t5-small on every field (solvent 20.9% top-5
against 49.5%, catalyst 0.6% against 31.5%, 72.1% of generations parseable against 93.2%). Three
causes, none of them the base, all fixed here:

1. **Decoding.** Every character `ensure_full_char_coverage` adds is decoded as its own segment
   and SentencePiece re-prefixes the segment after it, so `[Pd]` came back as `[Pd ]` and RDKit
   rejected it — 1970 of 5687 top-1 generations carried such a space. `run_reactiont5_topk.py:189`
   had always stripped spaces for Model 1; the conditions evaluator now does the same. Re-parsing
   the finished run's own generations with the strip lifted catalyst top-5 from 0.6% to 17.5%,
   with no retraining: that share of the loss was measurement, not the model.
2. **Truncation.** `--max-target-length 64` cut 4.2% of training targets mid-SMILES (catalysts run
   to 187 tokens at p99), teaching the model that a target may simply stop; 1584 of 5687
   generations came back with fewer than four fields. Now 200, which truncates 0.4%.
3. **Budget.** Training stopped at epoch 1.728 of 3 when the 150-minute cap hit, with `eval_loss`
   still falling monotonically (4.474 → 1.182, no upturn). That is the part the other two fixes do
   not explain, and it is why the budget is now 270 minutes — measured at ~87 min/epoch.

**Reference points, same 5,687-record clean test** (strict top-1 / strict top-5 / relaxed top-1 /
relaxed top-5): solvent 29.5 / 49.5 / 38.8 / 61.3; catalyst 22.3 / 31.5 / 27.8 / 39.8;
temperature 3.6 / 14.9 / 4.4 / 17.0; yield 0.6 / 5.6 / 0.8 / 7.4; parse rate 93.2%. Untrained
t5-small scores 0.0% on every field with 0.0% parseable output.

**Data:** `kuzmenkooleh/retro-planner-ord-conditions-300k` (138,869 train + 7,714 val) and
`kuzmenkooleh/retro-planner-ord-conditions-test-clean` (5,687 records, leak-free).

**Cost:** ~4 h 30 min training + ~40 min evaluation. Needs a fresh weekly GPU quota.

**Before running:** Settings -> **Internet** on, **GPU T4 x2** on. Run as **Save & Run All (Commit)**.

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available(), "| devices:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  Device {i}:", torch.cuda.get_device_name(i))

In [ ]:
import os
if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
import glob

train_file = next(glob.iglob("/kaggle/input/**/conditions_train.jsonl", recursive=True))
val_file = next(glob.iglob("/kaggle/input/**/conditions_val.jsonl", recursive=True))
test_file = next(glob.iglob("/kaggle/input/**/v2_ord_conditions_test_clean.jsonl", recursive=True))
for path in (train_file, val_file, test_file):
    print(path, sum(1 for _ in open(path)), "rows")

base_model = "sagawa/CompoundT5"
learning_rate = 5e-4   # same as the t5-small 138k run
output_dir = "/kaggle/working/model2_conditions_compoundt5"
# Sized against 5 h of remaining weekly quota, not against the ideal run. The first
# attempt covered 1.728 epochs in 150 minutes, i.e. ~87 min/epoch, so 225 minutes reaches
# roughly epoch 2.6 of 3 and leaves the ~40-minute evaluation inside the quota with margin.
# Being cut mid-run would cost the whole session, so the cap is the binding constraint here;
# `load_best_model_at_end` keeps the best checkpoint wherever training stops.
time_budget_minutes = 225

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)
log_path = f"{output_dir}/train.log"

# Batch 8 x 4 accumulation, not the script's default 32 x 1: that default was set for
# t5-small (60M) and CompoundT5 is 220M, which put both T4s at 14.55 of 14.56 GiB and
# raised `torch.OutOfMemoryError` a few steps in. The effective batch stays 32, so the
# optimization is unchanged from the t5-small runs -- only the memory footprint moves.
#
# --max-target-length 200, not 64: catalysts are whole organometallic complexes, and
# under CompoundT5's vocabulary the compact target runs to 187 tokens at p99 and 287 at
# most. A 64-token cap truncated 4.2% of training targets mid-SMILES, and the model
# learned to emit targets that simply stop -- 1584 of 5687 generations in the first
# attempt came back with fewer than four fields and could not be parsed at all. 200
# leaves 0.4% truncated; padding is dynamic, so the shorter 50% (16 tokens) costs
# nothing extra.
!torchrun --nproc_per_node=2 scripts/train_conditions_model.py \
    --base-model "{base_model}" \
    --train-file "{train_file}" \
    --val-file "{val_file}" \
    --output-dir "{output_dir}" \
    --local-work-dir /kaggle/temp/local_model2_work \
    --target-format compact \
    --max-target-length 200 \
    --per-device-train-batch-size 8 \
    --per-device-eval-batch-size 8 \
    --gradient-accumulation-steps 4 \
    --learning-rate {learning_rate} \
    --num-train-epochs 3 \
    --time-budget-minutes {time_budget_minutes} \
    > "{log_path}" 2>&1
print("training done; tail of log:")
!tail -5 "{log_path}"

In [ ]:
# The vocabulary repair must have fired: CompoundT5's 221-token ZINC vocabulary cannot spell
# `|`, `?` or several digits. Without it the targets would be <unk>-corrupted, which is exactly
# what sank the earlier ReactionT5-based attempt.
!grep -E "new character token|Train examples" "{log_path}"

import json
from transformers import AutoTokenizer

saved_tokenizer = AutoTokenizer.from_pretrained(f"{output_dir}/final")
embedding_rows = json.load(open(f"{output_dir}/final/config.json"))["vocab_size"]
print("tokenizer length:", len(saved_tokenizer), "| embedding rows:", embedding_rows)
assert len(saved_tokenizer) == embedding_rows, "tokenizer and embedding matrix disagree"
print("format marker:", open(f"{output_dir}/final/conditions_format.json").read())

In [ ]:
import json

state = json.load(open(f"{output_dir}/latest_checkpoint/trainer_state.json"))
points = [(h["epoch"], h["eval_loss"]) for h in state["log_history"] if "eval_loss" in h]
for epoch, loss in points[:: max(1, len(points) // 10)]:
    print(f"  epoch {epoch:5.2f}  eval_loss {loss:.4f}")
print("  last:", points[-1], "| best:", state.get("best_metric"))

In [ ]:
# --target-format defaults to auto, which reads the marker written next to the checkpoint.
# Batch 8 for the same reason as training: beam 10 over a 220M model holds 10 sequences per
# record in memory, where the t5-small runs used 32. Generation length matches training.
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{output_dir}/final" \
    --test-file "{test_file}" \
    --num-beams 10 --batch-size 8 --device cuda \
    --max-target-length 200 \
    --output "/kaggle/working/B1_conditions_compoundt5_compact_clean_topk.json"

In [ ]:
import json
data = json.load(open("/kaggle/working/B1_conditions_compoundt5_compact_clean_topk.json"))
summary = data["summary"] if "summary" in data else data
print(json.dumps(summary, indent=2))